# 02 — EDA: Meat Demand Patterns & Waste Drivers

**Business question:** When does FreshMart's meat department waste the most, and what drives it?

**Dataset:** `clean_freshmart_data.csv` — 100,000 records (2023–2024), 50 stores × 5 US regions.

This notebook:
1. Compares waste across product categories (is Meat the worst?)
2. Finds day-of-week demand and waste patterns for Meat
3. Tests whether markdowns actually reduce waste
4. Identifies spoilage drivers (temperature, days to expiry)
5. Quantifies the profit impact

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')

df = pd.read_csv('/content/drive/MyDrive/포트폴리오-FreshMart(캐글)/clean_freshmart_data.csv')
print('Shape:', df.shape)

## 1. Waste by category

**Why:** Before zooming into Meat, we check whether Meat is actually the problem child — or if another category wastes more.

In [ ]:
cat_waste = df.groupby('category').agg(
    records=('record_id', 'count'),
    avg_waste_pct=('waste_pct', 'mean'),
    spoilage_rate=('was_spoiled', 'mean'),
    total_waste_cost=('waste_cost', 'sum'),
    total_profit=('profit', 'sum'),
)
print(cat_waste.round(2).sort_values('avg_waste_pct', ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=cat_waste.reset_index().sort_values('avg_waste_pct', ascending=False),
            x='category', y='avg_waste_pct', ax=ax, palette='Reds_r')
ax.set_title('Average Waste % by Category')
ax.set_ylabel('Avg Waste %')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 2. Meat: day-of-week demand vs waste

**Why:** This is the core business question. If waste spikes on specific weekdays, ordering can be tuned per weekday — the exact lever a meat department manager controls.

In [ ]:
meat = df[df['category'] == 'Meat'].copy()
print(f'Meat records: {len(meat):,} ({len(meat)/len(df)*100:.1f}% of all)')

dow = meat.groupby('day_of_week').agg(
    avg_units_sold=('units_sold', 'mean'),
    avg_waste_pct=('waste_pct', 'mean'),
    avg_initial_qty=('initial_quantity', 'mean'),
)
dow.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
print(dow.round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dow['avg_units_sold'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Meat: Avg Units Sold by Weekday')
axes[0].set_ylabel('Units Sold')

dow['avg_waste_pct'].plot(kind='bar', ax=axes[1], color='indianred')
axes[1].set_title('Meat: Avg Waste % by Weekday')
axes[1].set_ylabel('Waste %')

plt.tight_layout()
plt.show()

**Read it like a manager:** Do high-order days match high-demand days? If waste peaks on a low-demand weekday, that day is over-ordered.

In [ ]:
# Over-order gap: ordered vs sold by weekday
dow['over_order_gap'] = dow['avg_initial_qty'] - dow['avg_units_sold']
print(dow[['avg_initial_qty', 'avg_units_sold', 'over_order_gap', 'avg_waste_pct']].round(1))

## 3. Do markdowns reduce waste?

**Why:** Discounting near-expiry meat is the standard waste lever. We test whether it actually works in this data — and at what cost to margin.

In [ ]:
md = meat.groupby('markdown_applied').agg(
    records=('record_id', 'count'),
    avg_waste_pct=('waste_pct', 'mean'),
    spoilage_rate=('was_spoiled', 'mean'),
    avg_discount=('discount_pct', 'mean'),
    avg_margin=('profit_margin_pct', 'mean'),
)
md.index = ['No markdown', 'Markdown']
print(md.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=meat, x='markdown_applied', y='waste_pct', ax=ax)
ax.set_xticklabels(['No markdown', 'Markdown'])
ax.set_title('Meat: Waste % Distribution — Markdown vs No Markdown')
ax.set_ylabel('Waste %')
plt.show()

## 4. Spoilage drivers

**Why:** If temperature abuse or short remaining shelf life predicts spoilage, the fix is operational (cold chain, rotation) — not just ordering less.

In [ ]:
drivers = meat.groupby('was_spoiled').agg(
    avg_temp_dev=('temp_deviation', 'mean'),
    avg_days_to_expiry=('days_until_expiry', 'mean'),
    avg_handling=('handling_score', 'mean'),
    avg_daily_demand=('daily_demand', 'mean'),
)
drivers.index = ['Not spoiled', 'Spoiled']
print(drivers.round(2))

## 5. Profit impact

**Why:** The business case needs dollars. How much profit does Meat waste destroy?

In [ ]:
meat_profit = meat['profit'].sum()
meat_waste_cost = meat['waste_cost'].sum()
all_profit = df['profit'].sum()

print(f"Meat total profit:      ${meat_profit:,.0f}")
print(f"Meat total waste cost:  ${meat_waste_cost:,.0f}")
print(f"Waste cost as % of meat profit: {meat_waste_cost/meat_profit*100:.1f}%")
print(f"Meat share of company profit:  {meat_profit/all_profit*100:.1f}%")